#  Coleta e Modelagem — Brasileirão Série A  2024

**Pergunta central:** o time que "merece" vencer, baseado no seu desempenho ofensivo/defensivo
histórico, é realmente o que mais pontua? Ou existem times sistematicamente "azarados"
(joga bem, pontua pouco) e "sortudos" (joga mal, pontua muito)?

**Fonte de dados:** [adaoduque/Brasileirao_Dataset](https://github.com/adaoduque/Brasileirao_Dataset)
— dataset público e gratuito, com todas as partidas do Brasileirão Série A desde 2003.

**Metodologia:** Construí um próprio "desempenho esperado" a partir do histórico de gols
de cada time, usando o mesmo princípio do modelo de Poisson (base do modelo Dixon-Coles,
usado profissionalmente em análise esportiva antes do xG se popularizar).


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

DATA_URL = "https://raw.githubusercontent.com/adaoduque/Brasileirao_Dataset/master/campeonato-brasileiro-full.csv"
SEASON = 2024  # última temporada completa disponível no dataset

df = pd.read_csv(DATA_URL)
df = df.rename(columns={"rodata": "rodada"})  # corrige typo do nome original da coluna
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year

df_season = df[df["ano"] == SEASON].copy()
print(f"Partidas na temporada {SEASON}: {len(df_season)}")
df_season.head()


In [ ]:
import os
os.chdir("/workspaces/analise-brasileirao")
print("Diretório atual:", os.getcwd())

## 1. Pontos reais

Calculamos os pontos que cada time realmente conquistou, a partir do placar de cada partida.

In [ ]:
def pontos_mandante(row):
    if row["mandante_Placar"] > row["visitante_Placar"]:
        return 3
    elif row["mandante_Placar"] == row["visitante_Placar"]:
        return 1
    return 0

def pontos_visitante(row):
    if row["visitante_Placar"] > row["mandante_Placar"]:
        return 3
    elif row["visitante_Placar"] == row["mandante_Placar"]:
        return 1
    return 0

df_season["pontos_mandante"] = df_season.apply(pontos_mandante, axis=1)
df_season["pontos_visitante"] = df_season.apply(pontos_visitante, axis=1)

pontos_casa = df_season.groupby("mandante")["pontos_mandante"].sum()
pontos_fora = df_season.groupby("visitante")["pontos_visitante"].sum()
pontos_reais = (pontos_casa.add(pontos_fora, fill_value=0)).sort_values(ascending=False)
pontos_reais.name = "pontos_reais"
pontos_reais.to_frame()


## 2. Índices de ataque e defesa por time

Para cada time, calculamos o quanto ele marca/sofre em casa e fora, relativo à média da liga.
Um índice de ataque > 1 significa que o time marca mais que a média; um índice de defesa > 1
significa que ele sofre mais gols que a média (ou seja, defesa pior).

In [ ]:
media_gols_mandante = df_season["mandante_Placar"].mean()
media_gols_visitante = df_season["visitante_Placar"].mean()

times = sorted(set(df_season["mandante"]) | set(df_season["visitante"]))

forca = {}
for time in times:
    jogos_casa = df_season[df_season["mandante"] == time]
    jogos_fora = df_season[df_season["visitante"] == time]

    forca[time] = {
        "ataque_casa": jogos_casa["mandante_Placar"].mean() / media_gols_mandante,
        "defesa_casa": jogos_casa["visitante_Placar"].mean() / media_gols_visitante,
        "ataque_fora": jogos_fora["visitante_Placar"].mean() / media_gols_visitante,
        "defesa_fora": jogos_fora["mandante_Placar"].mean() / media_gols_mandante,
    }

df_forca = pd.DataFrame(forca).T
df_forca.sort_values("ataque_casa", ascending=False)


## 3. Gols esperados por partida

Para cada partida da temporada, estimamos quantos gols cada time "deveria" fazer,
combinando o ataque de um lado com a defesa do outro.

In [ ]:
def gols_esperados(row):
    mandante, visitante = row["mandante"], row["visitante"]
    xg_mandante = media_gols_mandante * df_forca.loc[mandante, "ataque_casa"] * df_forca.loc[visitante, "defesa_fora"]
    xg_visitante = media_gols_visitante * df_forca.loc[visitante, "ataque_fora"] * df_forca.loc[mandante, "defesa_casa"]
    return pd.Series({"xg_mandante": xg_mandante, "xg_visitante": xg_visitante})

df_season[["xg_mandante", "xg_visitante"]] = df_season.apply(gols_esperados, axis=1)
df_season[["mandante", "visitante", "mandante_Placar", "visitante_Placar", "xg_mandante", "xg_visitante"]].head(10)


## 4. Convertendo gols esperados em pontos esperados

Usei a distribuição de Poisson para transformar os gols esperados de cada time em
probabilidades de vitória, empate e derrota, e daí em "pontos esperados" para a partida.



In [ ]:
MAX_GOLS = 10

def pontos_esperados_partida(xg_mandante, xg_visitante):
    prob_gols_mandante = [poisson.pmf(k, xg_mandante) for k in range(MAX_GOLS + 1)]
    prob_gols_visitante = [poisson.pmf(k, xg_visitante) for k in range(MAX_GOLS + 1)]

    p_vitoria_mandante = 0.0
    p_empate = 0.0
    p_vitoria_visitante = 0.0

    for gm in range(MAX_GOLS + 1):
        for gv in range(MAX_GOLS + 1):
            p = prob_gols_mandante[gm] * prob_gols_visitante[gv]
            if gm > gv:
                p_vitoria_mandante += p
            elif gm == gv:
                p_empate += p
            else:
                p_vitoria_visitante += p

    pontos_esperados_mandante = p_vitoria_mandante * 3 + p_empate * 1
    pontos_esperados_visitante = p_vitoria_visitante * 3 + p_empate * 1
    return pontos_esperados_mandante, pontos_esperados_visitante

resultado = df_season.apply(
    lambda row: pontos_esperados_partida(row["xg_mandante"], row["xg_visitante"]),
    axis=1,
    result_type="expand",
)
df_season[["pontos_esperados_mandante", "pontos_esperados_visitante"]] = resultado
df_season[["mandante", "visitante", "pontos_mandante", "pontos_esperados_mandante",
           "pontos_visitante", "pontos_esperados_visitante"]].head(10)


## 5. O índice de sorte/azar

Somamos os pontos esperados de cada time ao longo da temporada e comparamos com os pontos reais.

In [ ]:
esperados_casa = df_season.groupby("mandante")["pontos_esperados_mandante"].sum()
esperados_fora = df_season.groupby("visitante")["pontos_esperados_visitante"].sum()
pontos_esperados_total = (esperados_casa.add(esperados_fora, fill_value=0))
pontos_esperados_total.name = "pontos_esperados"

df_comparacao = pd.concat([pontos_reais, pontos_esperados_total], axis=1)
df_comparacao["indice_sorte"] = df_comparacao["pontos_reais"] - df_comparacao["pontos_esperados"]
df_comparacao = df_comparacao.sort_values("indice_sorte", ascending=False)
df_comparacao.round(1)


## 6. Visualizações

Dois gráficos para contar a história: um mostrando cada time no espaço "esperado vs. real",
e outro rankeando quem teve mais sorte e mais azar na temporada.

In [ ]:
import os
print("Diretório atual do notebook:", os.getcwd())
print("Existe 'reports'?", os.path.exists("reports"))
print("Existe 'reports/figures'?", os.path.exists("reports/figures"))
print(os.listdir("."))

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"
COR_SORTE = "#2E86AB"
COR_AZAR = "#E63946"
COR_NEUTRO = "#6C757D"

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

lim_min = min(df_comparacao["pontos_esperados"].min(), df_comparacao["pontos_reais"].min()) - 5
lim_max = max(df_comparacao["pontos_esperados"].max(), df_comparacao["pontos_reais"].max()) + 5

ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", color=COR_NEUTRO,
        linewidth=1.5, zorder=1, label="Linha de igualdade\n(sorte neutra)")

cores_pontos = [COR_SORTE if v >= 0 else COR_AZAR for v in df_comparacao["indice_sorte"]]
ax.scatter(df_comparacao["pontos_esperados"], df_comparacao["pontos_reais"], s=180,
           c=cores_pontos, alpha=0.85, edgecolors="white", linewidths=1.5, zorder=3)

for time, row in df_comparacao.iterrows():
    ax.annotate(time, (row["pontos_esperados"], row["pontos_reais"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)

ax.set_xlabel("Pontos Esperados (baseado em desempenho ofensivo/defensivo)", fontsize=12, labelpad=10)
ax.set_ylabel("Pontos Reais (tabela final)", fontsize=12, labelpad=10)
ax.set_title(f"Quem teve sorte e quem teve azar no Brasileirão {SEASON}?", fontsize=15, fontweight="bold", pad=25)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.25, linewidth=0.5)
ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.legend(loc="lower right", frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig("reports/figures/01_dispersao_pontos.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

df_plot = df_comparacao.sort_values("indice_sorte")
cores_barras = [COR_SORTE if v >= 0 else COR_AZAR for v in df_plot["indice_sorte"]]

bars = ax.barh(df_plot.index, df_plot["indice_sorte"], color=cores_barras,
               height=0.65, edgecolor="white", linewidth=0.5)

for bar, valor in zip(bars, df_plot["indice_sorte"]):
    x_pos = valor + (0.15 if valor >= 0 else -0.15)
    ha = "left" if valor >= 0 else "right"
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f"{valor:+.1f}",
            va="center", ha=ha, fontsize=9, fontweight="bold")

ax.axvline(0, color="#212529", linewidth=1)
ax.set_xlabel("Índice de Sorte/Azar (pontos reais − pontos esperados)", fontsize=12, labelpad=10)
ax.set_title(f"Ranking de Sorte e Azar — Brasileirão {SEASON}", fontsize=15, fontweight="bold", pad=15)

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(True, axis="x", alpha=0.25, linewidth=0.5)
ax.tick_params(left=False)

plt.tight_layout()
plt.savefig("reports/figures/02_ranking_sorte_azar.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. Validação: a sorte persiste entre temporadas?

Se o "Índice de Sorte/Azar" capturasse algo estrutural sobre o time (e não apenas
variância aleatória de uma temporada), esperaríamos que times sortudos/azarados
se mantivessem consistentes ano após ano. Comparação com os anos de 2022, 2023 e 2024. 

In [ ]:
def calcular_indice_sorte(df, season):
    df_s = df[df["ano"] == season].copy()

    df_s["pontos_mandante"] = df_s.apply(pontos_mandante, axis=1)
    df_s["pontos_visitante"] = df_s.apply(pontos_visitante, axis=1)
    pontos_casa = df_s.groupby("mandante")["pontos_mandante"].sum()
    pontos_fora = df_s.groupby("visitante")["pontos_visitante"].sum()
    pontos_reais_s = pontos_casa.add(pontos_fora, fill_value=0)
    pontos_reais_s.name = "pontos_reais"

    media_gm = df_s["mandante_Placar"].mean()
    media_gv = df_s["visitante_Placar"].mean()
    times_s = sorted(set(df_s["mandante"]) | set(df_s["visitante"]))

    forca_s = {}
    for time in times_s:
        jc = df_s[df_s["mandante"] == time]
        jf = df_s[df_s["visitante"] == time]
        forca_s[time] = {
            "ataque_casa": jc["mandante_Placar"].mean() / media_gm,
            "defesa_casa": jc["visitante_Placar"].mean() / media_gv,
            "ataque_fora": jf["visitante_Placar"].mean() / media_gv,
            "defesa_fora": jf["mandante_Placar"].mean() / media_gm,
        }
    df_forca_s = pd.DataFrame(forca_s).T

    def gols_esperados_s(row):
        m, v = row["mandante"], row["visitante"]
        xgm = media_gm * df_forca_s.loc[m, "ataque_casa"] * df_forca_s.loc[v, "defesa_fora"]
        xgv = media_gv * df_forca_s.loc[v, "ataque_fora"] * df_forca_s.loc[m, "defesa_casa"]
        return pd.Series({"xg_mandante": xgm, "xg_visitante": xgv})

    df_s[["xg_mandante", "xg_visitante"]] = df_s.apply(gols_esperados_s, axis=1)

    res_s = df_s.apply(
        lambda r: pontos_esperados_partida(r["xg_mandante"], r["xg_visitante"]),
        axis=1, result_type="expand"
    )
    df_s[["pontos_esperados_mandante", "pontos_esperados_visitante"]] = res_s

    esperados_casa = df_s.groupby("mandante")["pontos_esperados_mandante"].sum()
    esperados_fora = df_s.groupby("visitante")["pontos_esperados_visitante"].sum()
    pontos_esperados_total_s = esperados_casa.add(esperados_fora, fill_value=0)
    pontos_esperados_total_s.name = "pontos_esperados"

    resultado = pd.concat([pontos_reais_s, pontos_esperados_total_s], axis=1)
    resultado["indice_sorte"] = resultado["pontos_reais"] - resultado["pontos_esperados"]
    resultado["temporada"] = season
    return resultado.sort_values("indice_sorte", ascending=False)

In [ ]:
resultados_temporadas = {}
for season in [2022, 2023, 2024]:
    resultados_temporadas[season] = calcular_indice_sorte(df, season)

df_todas_temporadas = pd.concat(resultados_temporadas.values())
pivot_sorte = df_todas_temporadas.pivot_table(
    index=df_todas_temporadas.index, columns="temporada", values="indice_sorte"
)

pivot_completo = pivot_sorte.dropna()
pivot_completo.round(1)

In [ ]:
correlacao = pivot_completo.corr().round(2)
print("Correlação do Índice de Sorte/Azar entre temporadas:")
correlacao